In [5]:
import polars as pl
import torch
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
)

from src.config import PROJECT_ROOT, PROCESSED_DATA_DIR
from src.data import DatasetLoader, Preprocessor
from src.db import PBWarehouse

In [2]:
dataset_loader = DatasetLoader(PBWarehouse())
data: pl.DataFrame = dataset_loader.load_dataset()
preprocessor = Preprocessor(data=data)
data = preprocessor.preprocess()
data

✅ Loading cached dataset - Done.
✅ Loading dataset - Done.
Categorizing source column...

/tmp/ipykernel_754485/2238835076.py:4: DeprecationWarning: preprocess is deprecated. Please call (__call__) the Preprocessor instance after instantiation instead.
  data = preprocessor.preprocess()


✅ Categorizing source column - Done.
✅ Preprocessing data - Done.


bookmark_count,favorite_count,favourites_count,follower_count,following_count,friends,is_blue_verified,is_hateful,listed_count,number_of_tweets,quote_count,reply_count,retweet_count,source,tweet_id,user_id,views
i64,i64,i64,i64,i64,i64,bool,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
0,0,941,39,6,6,false,1,0,422,0,0,0,0,1269916756719685632,1264846057906806786,0
23,4860,13269,95956,1596,1595,true,1,245,21715,28,334,156,0,1277976913743503365,23719684,0
0,2,1494,1394,427,427,false,1,30,1173,0,0,0,0,1285714950896443395,1081973336010309632,0
0,4,2675,49,165,165,false,1,3,1866,0,1,0,0,1284209535561818112,190751291,0
0,2,6,214664,258,258,false,1,0,310173,1,7,1,0,1279563405536325632,121639467,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0,1,2447,468,979,0,true,1,3,4284,0,1,1,0,1269054693323530241,426170736,0
14,3034,532,30305,234,233,false,1,217,2069,86,40,1378,0,1274010135606620161,2904953395,0
0,0,3071,54046,52576,0,true,1,382,28706,0,0,1,0,1284986922490720257,14870085,0


In [7]:
import polars as pl

finetuned_data = pl.read_csv(PROCESSED_DATA_DIR / "hatebert-finetuned.csv")
finetuned_data

text,tweet_id,is_hateful
str,i64,i64
"""@TRCdocumentary @Blklivesmatte…",1269916756719685632,1
"""@terrycrews @jonathanwsabin Th…",1277976913743503365,1
"""Join Bernie and Sunrise leader…",1285714950896443395,1
"""@prageru It's not the women's …",1284209535561818112,1
"""A #BlackLivesMatter protest in…",1279563405536325632,1
…,…,…
"""I curated a list of black-owne…",1269054693323530241,1
"""Defund. Disarm. Dismantle. #De…",1274010135606620161,1
"""Bill Nye with the mic drop. #S…",1284986922490720257,1


In [8]:
for row in finetuned_data.iter_rows(named=True):
    data = data.with_columns(
        pl.when(pl.col("tweet_id") == row["tweet_id"])
          .then(pl.lit(row["is_hateful"]))
          .otherwise(pl.col("is_hateful"))
          .alias("is_hateful")
    )

data.write_csv(PROCESSED_DATA_DIR / "finetuned_dataset_cache.csv")

In [9]:
data

bookmark_count,favorite_count,favourites_count,follower_count,following_count,friends,is_blue_verified,is_hateful,listed_count,number_of_tweets,quote_count,reply_count,retweet_count,source,tweet_id,user_id,views
i64,i64,i64,i64,i64,i64,bool,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
0,0,941,39,6,6,false,1,0,422,0,0,0,0,1269916756719685632,1264846057906806786,0
23,4860,13269,95956,1596,1595,true,1,245,21715,28,334,156,0,1277976913743503365,23719684,0
0,2,1494,1394,427,427,false,1,30,1173,0,0,0,0,1285714950896443395,1081973336010309632,0
0,4,2675,49,165,165,false,1,3,1866,0,1,0,0,1284209535561818112,190751291,0
0,2,6,214664,258,258,false,1,0,310173,1,7,1,0,1279563405536325632,121639467,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
0,1,2447,468,979,0,true,1,3,4284,0,1,1,0,1269054693323530241,426170736,0
14,3034,532,30305,234,233,false,1,217,2069,86,40,1378,0,1274010135606620161,2904953395,0
0,0,3071,54046,52576,0,true,1,382,28706,0,0,1,0,1284986922490720257,14870085,0


In [3]:
MODEL_NAME = "Hate-speech-CNERG/bert-base-uncased-hatexplain"
# HATEBERT on Hugging Face

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, attn_implementation="eager"
)

model.load_state_dict(torch.load(PROJECT_ROOT / "finetuned_model-32-epochs.pth"))
model.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [ ]:
df.with_columns(
    pl.col("text").map_elements(lambda x: classify_text(x), return_dtype=pl.Int8).alias("is_hateful")
)